# DeBERTa-v3-small (PRETRAINED TRANSFORMER)

In [1]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers==4.40.0', 'sentencepiece'], check=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


CompletedProcess(args=['pip', 'install', '-q', 'transformers==4.40.0', 'sentencepiece'], returncode=0)

In [2]:
import os, warnings, pickle
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

INPUT_DIR = Path('/kaggle/input/competitions/smart-mcq-solver-challenge')
OUTPUT_DIR = Path('/kaggle/working')

Device : cuda
GPU    : Tesla T4


In [4]:
# creating logs and models folder in kaggle 

OUTPUT_DIR = Path('/kaggle/working')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

(OUTPUT_DIR / "logs").mkdir(
    parents=True,
    exist_ok=True
)

(OUTPUT_DIR / "models").mkdir(
    parents=True,
    exist_ok=True
)

In [5]:
# config define randomly ...

CFG = dict(
    model_name   = 'microsoft/deberta-v3-small',
    max_len      = 320,    # tokens for (prompt + ONE option)
    batch_size   = 32,    
    lr           = 2e-5,   # transformer learning rate
    weight_decay = 0.01,
    epochs       = 5,      # transformers converge fast
    warmup_ratio = 0.1,    # 10% of steps for warmup
    patience     = 3,
    seed         = 42,
)

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

print('Config:')
for k, v in CFG.items(): print(f'  {k:<14}: {v}')

Config:
  model_name    : microsoft/deberta-v3-small
  max_len       : 320
  batch_size    : 32
  lr            : 2e-05
  weight_decay  : 0.01
  epochs        : 5
  warmup_ratio  : 0.1
  patience      : 3
  seed          : 42
